# 📁 Synthetic Data Generator

|             |                                              |
|-------------|----------------------------------------------|
|Notebook     |`01_synthetic_data_generator-ipynb`           |
|Version      |`v0.1`                                        |
|Date         |30/04/2026                                    |
|Project      |GLMs vs Neural Networks in Practice           |
|Author       |Carlos Arocha                                 |

---

### Purpose
This notebook generates:
- `idi_exposure_synth_v1.csv` (one row per insured per month)
- `idi_claims_synth_v1.csv` (one row per claim)

Design goals:
- Reproducible (fixed seeds)
- Safe to share (fully synthetic)
- Governance-friendly (data dictionary + acceptance checks)


## Setup

In [1]:
# Mount Google Drive
#from google.colab import drive
# drive.mount("/content/drive")

# Required libraries
from pathlib import Path
import os
from hashlib import sha256

import numpy as np
import pandas as pd

In [2]:
# Load configuration

PROJECT_ROOT = Path("C:\\Users\\ca\\OneDrive\\Documents\\@ CA\\@ PROJECTS\\FFNNs\\IDS 2026")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

# Core folders
DATA_PATH = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
PROCESSED_DATA_PATH = DATA_PATH / "processed"
INTERIM_DATA_PATH = DATA_PATH / "interim"

for folder in [
    DATA_PATH,
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    INTERIM_DATA_PATH,
]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root set to: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")

Project root set to: C:\Users\ca\OneDrive\Documents\@ CA\@ PROJECTS\FFNNs\IDS 2026
Current working directory: C:\Users\ca\OneDrive\Documents\@ CA\@ PROJECTS\FFNNs\IDS 2026


In [3]:
# Parameters
PARAMS = dict(
    n_lives=6000,
    n_months=24,
    start="2022-01-01",
    n_region=120,
    n_occ=180,
    n_employer=350,
    n_plan=12,
    emb_dim=6,
    seed=6502
)

## Synthetic Generator

Key modeling structure (useful for GLM vs FFNN comparisons):
- high-cardinality categoricals: region_id, occ_id, employer_id, plan_id
- nonlinear hazard thresholds: ReLU(heat - thr), ReLU(smoke - thr)
- multi-way interactions: hazard × vulnerability × age × occupation profile × latent factors
- severity mixture regime: some claims are "complex/long-duration" with heavier tail


In [4]:
# Synthetic generator (expo + claims)

def generate_synth_idi_expo_claims(

    n_lives: int = 10_000,
    n_months: int = 24,
    start: str = "2026-01-01",
    n_region: int = 120,
    n_occ: int = 180,
    n_employer: int = 350,
    n_plan: int = 12,
    emb_dim: int = 6,
    seed: int = 6502

):
    rng = np.random.default_rng(seed)

    # Latent embeddings (true structure FFNN embeddings can learn)
    E_region = rng.normal(0, 1, size=(n_region, emb_dim))
    E_occ    = rng.normal(0, 1, size=(n_occ, emb_dim))
    E_emp    = rng.normal(0, 1, size=(n_employer, emb_dim))
    E_plan   = rng.normal(0, 1, size=(n_plan, emb_dim))

    # Region vulnerability & baseline propensity
    region_heat_vuln  = rng.uniform(0.8, 1.3, size=n_region)
    region_smoke_vuln = rng.uniform(0.7, 1.4, size=n_region)
    region_base       = rng.normal(0, 0.25, size=n_region)

    # Occupation profiles (latent drivers)
    occ_phys  = rng.uniform(0.0, 1.0, size=n_occ)   # physical vulnerability
    occ_psych = rng.uniform(0.0, 1.0, size=n_occ)   # psych/stress vulnerability proxy

    # Plan generosity multiplier (affects severity)
    plan_generosity = rng.uniform(0.7, 1.2, size=n_plan)

    # Life-level covariates (static)
    lives = pd.DataFrame({

        "life_id": np.arange(n_lives),
        "region_id": rng.integers(0, n_region, size=n_lives),
        "occ_id": rng.integers(0, n_occ, size=n_lives),
        "employer_id": rng.integers(0, n_employer, size=n_lives),
        "plan_id": rng.integers(0, n_plan, size=n_lives),
        "sex": rng.choice(["F","M"], size=n_lives, p=[0.48,0.52]),
        "smoker": rng.choice(["N","Y"], size=n_lives, p=[0.85,0.15]),
        "bmi_band": rng.choice(["normal","over","obese"], size=n_lives, p=[0.60,0.28,0.12]),
        "salary_band": rng.choice(["40-60K","60-90K","90-130K","130K+"], size=n_lives, p=[0.30,0.40,0.20,0.10]),
        "underwriting_class": rng.choice(["preferred","standard","substandard"], size=n_lives, p=[0.25,0.65,0.10]),
        "elim_period": rng.choice(["30","60","90"], size=n_lives, p=[0.55,0.30,0.15]),
        "benefit_period": rng.choice(["2y","5y","to65"], size=n_lives, p=[0.25,0.45,0.30]),
        "cola_flag": rng.integers(0, 2, size=n_lives),

    })

    lives["chronic_resp_flag"]   = (rng.random(n_lives) < 0.10).astype(int)
    lives["chronic_cardio_flag"] = (rng.random(n_lives) < 0.08).astype(int)
    lives["prior_claim_flag"]    = (rng.random(n_lives) < 0.03).astype(int)

    cap_map = {"40-60K": 4000, "60-90K": 6500, "90-130K": 9000, "130K+": 12000}
    lives["monthly_benefit_cap"] = lives["salary_band"].map(cap_map).astype(float)
    lives["base_age"] = rng.integers(28, 59, size=n_lives)

    # Months
    months = pd.date_range(start=start, periods=n_months, freq="MS")

    # Region-month hazards (seasonality + shocks)
    heat  = np.zeros((n_region, n_months), dtype=float)
    smoke = np.zeros((n_region, n_months), dtype=float)
    flood = np.zeros((n_region, n_months), dtype=int)

    for r in range(n_region):
        for t, m in enumerate(months):
            mm = int(m.month)
            season_heat  = 1.55 if mm in [6,7,8] else (0.85 if mm in [12,1,2] else 1.0)
            season_smoke = 1.35 if mm in [6,7,8] else 0.90

            # Rare shocks amplify hazards
            shock_heat  = 1.0 + (rng.random() < 0.06) * rng.uniform(0.25, 0.70)
            shock_smoke = 1.0 + (rng.random() < 0.05) * rng.uniform(0.30, 0.90)

            heat[r, t]  = season_heat  * region_heat_vuln[r]  * shock_heat  * rng.lognormal(0, 0.08)
            smoke[r, t] = season_smoke * region_smoke_vuln[r] * shock_smoke * rng.lognormal(0, 0.10)
            flood[r, t] = int(rng.random() < (0.012 + 0.010*region_smoke_vuln[r]/1.4))

    # Exposure-month rows
    rows = []
    start_dt = months[0]
    for t, m in enumerate(months):
        tmp = lives.copy()
        tmp["month_start"] = m
        tmp["calendar_year"] = int(m.year)
        tmp["month"] = int(m.month)
        tmp["season"] = "summer" if tmp["month"].iloc[0] in [6,7,8] else ("winter" if tmp["month"].iloc[0] in [12,1,2] else "shoulder")
        tmp["exposure_months"] = 1.0
        tmp["age"] = tmp["base_age"] + int(m.year - start_dt.year)

        rid = tmp["region_id"].values
        tmp["heat_index_30d"] = heat[rid, t]
        tmp["smoke_pm25_30d"] = smoke[rid, t]
        tmp["flood_flag_30d"] = flood[rid, t]

        tmp["compound_heat_smoke"] = ((tmp["heat_index_30d"] > 1.35) & (tmp["smoke_pm25_30d"] > 1.15)).astype(int)
        tmp["extreme_heat"] = (tmp["heat_index_30d"] > 1.55).astype(int)

        rows.append(tmp)

    expo = pd.concat(rows, ignore_index=True)

    # Incidence DGP (nonlinear + multi-way interactions + latent vectors)
    rid = expo["region_id"].values
    oid = expo["occ_id"].values
    eid = expo["employer_id"].values
    pid = expo["plan_id"].values

    h_lat = (E_region[rid] + 0.9*E_occ[oid] + 0.4*E_emp[eid] + 0.6*E_plan[pid])

    age_c  = (expo["age"].values - 40.0) / 10.0
    smoker = (expo["smoker"].values == "Y").astype(float)
    obese  = (expo["bmi_band"].values == "obese").astype(float)
    over   = (expo["bmi_band"].values == "over").astype(float)
    prior  = expo["prior_claim_flag"].values.astype(float)
    resp   = expo["chronic_resp_flag"].values.astype(float)
    cardio = expo["chronic_cardio_flag"].values.astype(float)
    cola   = expo["cola_flag"].values.astype(float)

    uw   = expo["underwriting_class"].map({"preferred":-0.20,"standard":0.0,"substandard":0.30}).values.astype(float)
    elim = expo["elim_period"].map({"30":0.12,"60":0.0,"90":-0.08}).values.astype(float)
    bp   = expo["benefit_period"].map({"2y":-0.08,"5y":0.0,"to65":0.10}).values.astype(float)

    heat30  = expo["heat_index_30d"].values
    smoke30 = expo["smoke_pm25_30d"].values
    flood30 = expo["flood_flag_30d"].values.astype(float)
    comp    = expo["compound_heat_smoke"].values.astype(float)
    xheat   = expo["extreme_heat"].values.astype(float)

    relu_heat  = np.maximum(0.0, heat30 - 1.25)
    relu_smoke = np.maximum(0.0, smoke30 - 1.05)

    occ_heat  = occ_phys[oid]
    occ_smoke = occ_psych[oid]

    z_base = (
        -5.20
        + 0.18*age_c + 0.28*smoker + 0.20*obese + 0.10*over
        + 0.40*prior + 0.20*resp + 0.18*cardio
        + uw + elim + bp
        + 0.10*cola
        + region_base[rid]
    )

    z_lat = (
        0.22*np.tanh(h_lat[:,0])
        + 0.18*np.tanh(h_lat[:,1]*h_lat[:,2])
        + 0.12*np.sin(h_lat[:,3])
    )

    z_haz = (
        0.55*relu_heat*(1.0 + 0.7*cardio + 0.5*age_c)*(0.6 + 1.0*occ_heat)
        + 0.48*relu_smoke*(1.0 + 0.9*resp + 0.35*occ_smoke)
        + 0.42*comp*(0.9 + 0.6*np.tanh(h_lat[:,4]))
        + 0.30*xheat*(0.8 + 0.8*np.tanh(h_lat[:,5]))
        + 0.22*flood30*(1.0 + 0.35*np.tanh(h_lat[:,1]))
    )

    lp = z_base + z_lat + z_haz
    p_onset = 1.0/(1.0 + np.exp(-lp))

    rng2 = np.random.default_rng(seed + 999)
    expo["onset"] = (rng2.random(len(expo)) < p_onset).astype(int)

    # Enforce <= 1 claim per life
    expo.sort_values(["life_id","month_start"], inplace=True)
    expo["cum"] = expo.groupby("life_id")["onset"].cumsum()
    expo.loc[expo["cum"] > 1, "onset"] = 0
    expo.drop(columns=["cum"], inplace=True)

    # Claims table
    claims = expo.loc[expo["onset"] == 1].copy().reset_index(drop=True)
    claims["claim_id"] = np.arange(len(claims))
    claims["onset_date"] = claims["month_start"]

    expo["claim_id"] = np.nan
    expo.loc[expo["onset"] == 1, "claim_id"] = claims["claim_id"].values

    # Severity DGP
    rid_c = claims["region_id"].values
    oid_c = claims["occ_id"].values
    eid_c = claims["employer_id"].values
    pid_c = claims["plan_id"].values

    h_lat_c = (E_region[rid_c] + 0.9*E_occ[oid_c] + 0.4*E_emp[eid_c] + 0.6*E_plan[pid_c])

    heat_c  = claims["heat_index_30d"].values
    smoke_c = claims["smoke_pm25_30d"].values
    relu_heat_c  = np.maximum(0.0, heat_c - 1.25)
    relu_smoke_c = np.maximum(0.0, smoke_c - 1.05)
    comp_c = claims["compound_heat_smoke"].values.astype(float)

    age_c2 = (claims["age"].values - 40.0) / 10.0
    cap = claims["monthly_benefit_cap"].values
    plan_g = plan_generosity[pid_c]

    mix_lp = (
        -0.2
        + 0.85*np.tanh(h_lat_c[:,2])
        + 0.60*comp_c
        + 0.35*claims["chronic_resp_flag"].values.astype(float)
        + 0.25*age_c2
    )    
    mix_p = 1.0/(1.0 + np.exp(-mix_lp))
    mix = (rng2.random(len(claims)) < mix_p).astype(int)

    
    # ReLU(x) = max(0, x), a non-linear transformation 
        
    # this is the log of the expected claim size on the log scale. Expected dollar amount is exp(-log_mu)
    log_mu = (
        np.log(52000.0)  # sets the base claim severity at 52K        
        + 0.07*age_c2   # older claimants have higher severity; age_c2 is centered at 40 and scaled by 10 years -> a 10-year increase in age multiplies expected severity by exp(0.07) = 1.072
        + 0.10*(claims["underwriting_class"].values == "substandard").astype(float)  # True if substandard underwriting class, which increases severity by exp(0.10) = 1.105
        + 0.06*(claims["elim_period"].values == "30").astype(float) # True if 30-day elimination period, which increases severity by exp(0.06) = 1.062
        + 0.11*np.log1p(cap/1000.0) # True if monthly benefit cap is positive, which increases severity by exp(0.11) = 1.116
        + 0.12*np.tanh(h_lat_c[:,0])  # Nonlinear effect, which captures interactions of region, occupation, employer, and plan characteristics
        + 0.11*np.tanh(h_lat_c[:,1]*h_lat_c[:,3])  # ditto
        + 0.14*relu_heat_c*(0.4 + 0.9*occ_phys[oid_c])  # Nonlinear effect of heat index, amplified for occupations with higher physical vulnerability
        + 0.10*relu_smoke_c*(0.5 + 0.8*occ_psych[oid_c])  # Nonlinear effect of smoke index, amplified for occupations with higher psych/stress vulnerability
        + 0.16*comp_c*(0.8 + 0.5*np.tanh(h_lat_c[:,4]))  # Compound heat-smoke effect, amplified for certain latent profiles
        + 0.30*mix  # Mixture component that increases severity for a subset of claims, based on latent and observed characteristics
        + np.log(plan_g)  # plan generosity multiplier, which increases severity by exp(log(plan_g)) = plan_g
    )

    sigma = 0.24 + 0.07*mix + 0.04*relu_heat_c + 0.04*relu_smoke_c
    cost = np.exp(log_mu + rng2.normal(0, sigma))
    cost = np.clip(cost, 1500.0, 400000.0)

    claims["total_paid_ultimate"] = np.round(cost, 0)

    for df in (expo, claims):
        df["region"]   = df["region_id"].astype(str)
        df["occ"]      = df["occ_id"].astype(str)
        df["employer"] = df["employer_id"].astype(str)
        df["plan"]     = df["plan_id"].astype(str)

    return expo.reset_index(drop=True), claims.reset_index(drop=True)

## Generate Dataset

In [5]:
# Generate dataset
expo_df, claims_df = generate_synth_idi_expo_claims(**PARAMS)

print("Exposure rows:", f"{len(expo_df):,.0f}")
print("Claims rows   :", f"{len(claims_df):,.0f}")
print("Monthly onset rate:", round(float(expo_df["onset"].mean()), 6))
print("Mean claim cost:", f"{claims_df['total_paid_ultimate'].mean():,.0f}")
print("P90 claim cost:", f"{np.quantile(claims_df['total_paid_ultimate'], 0.90):,.0f}")

Exposure rows: 144,000
Claims rows   : 1,565
Monthly onset rate: 0.010868
Mean claim cost: 101,625
P90 claim cost: 160,396


## Data Dictionary

In [6]:
# Data dictionary

dd = [

  ("life_id", "int", "Unique insured identifier"),
  ("month_start", "date", "Month start date for exposure record"),
  ("exposure_months", "float", "Exposure in months"),
  ("claim_id", "int/NA", "Claim id if onset=1 in that month, else NA"),
  ("onset", "0/1", "Claim onset indicator for that month"),

  ("age", "int", "Age at month"),
  ("sex", "cat", "Sex (F/M)"),
  ("smoker", "cat", "Smoker status (N/Y)"),
  ("bmi_band", "cat", "BMI band (normal/over/obese)"),
  ("salary_band", "cat", "Salary band"),
  ("monthly_benefit_cap", "float", "Monthly benefit cap proxy"),
  ("underwriting_class", "cat", "UW class (preferred/standard/substandard)"),
  ("elim_period", "cat", "Elimination period (30/60/90 days)"),
  ("benefit_period", "cat", "Benefit period (2y/5y/to65)"),
  ("cola_flag", "0/1", "Cost-of-living adjustment flag"),

  ("chronic_resp_flag", "0/1", "Respiratory vulnerability proxy"),
  ("chronic_cardio_flag", "0/1", "Cardio vulnerability proxy"),
  ("prior_claim_flag", "0/1", "Prior claim history proxy"),

  ("region_id", "int", "Region id (0..n_region-1)"),
  ("occ_id", "int", "Occupation id (0..n_occ-1)"),
  ("employer_id", "int", "Employer id (0..n_employer-1)"),
  ("plan_id", "int", "Plan id (0..n_plan-1)"),
  ("region", "cat", "String version of region_id (convenience)"),
  ("occ", "cat", "String version of occ_id (convenience)"),
  ("employer", "cat", "String version of employer_id (convenience)"),
  ("plan", "cat", "String version of plan_id (convenience)"),

  ("heat_index_30d", "float", "Heat index proxy (region×month), seasonal + shocks"),
  ("smoke_pm25_30d", "float", "Smoke/PM2.5 proxy (region×month), seasonal + shocks"),
  ("flood_flag_30d", "0/1", "Flood event indicator"),
  ("compound_heat_smoke", "0/1", "Indicator: heat and smoke jointly elevated"),
  ("extreme_heat", "0/1", "Indicator: extreme heat episode"),

  ("onset_date", "date", "(claims table only) onset month"),
  ("total_paid_ultimate", "float", "(claims table only) ultimate paid amount"),

]

dd_df = pd.DataFrame(dd, columns=["field", "type", "description"])

dd_df.shape

(33, 3)

## Acceptance Checks

In [7]:
# Acceptance checks
checks = {}

checks["expo_rows"] = len(expo_df)
checks["claims_rows"] = len(claims_df)

inc_rate = float(expo_df["onset"].mean())
checks["monthly_incidence_rate"] = inc_rate

mean_cost = float(claims_df["total_paid_ultimate"].mean()) if len(claims_df) else float("nan")
p90_cost = float(np.quantile(claims_df["total_paid_ultimate"], 0.90)) if len(claims_df) else float("nan")
checks["mean_claim_cost"] = mean_cost
checks["p90_claim_cost"] = p90_cost

claims_per_life = expo_df.loc[expo_df["onset"] == 1].groupby("life_id")["onset"].sum()
checks["max_claims_per_life"] = int(claims_per_life.max()) if len(claims_per_life) else 0

checks["expo_missing_pct"] = float(expo_df.isna().mean().mean())
checks["claims_missing_pct"] = float(claims_df.isna().mean().mean())

# Acceptance bands
assert 0.003 <= inc_rate <= 0.05, f"Incidence rate out of expected range: {inc_rate}"
assert 20000 <= mean_cost <= 200000, f"Mean claim cost out of expected range: {mean_cost}"
assert checks["max_claims_per_life"] <= 1, f"More than 1 claim per life detected: {checks['max_claims_per_life']}"

## Save Outputs

In [8]:
# Save outputs
expo_path = RAW_DATA_PATH / "idi_exposure_synth_v1.csv"
claims_path = RAW_DATA_PATH / "idi_claims_synth_v1.csv"

expo_df.to_csv(expo_path, index=False)
claims_df.to_csv(claims_path, index=False)

print("Saved:", expo_path, "and", claims_path)

Saved: C:\Users\ca\OneDrive\Documents\@ CA\@ PROJECTS\FFNNs\IDS 2026\data\raw\idi_exposure_synth_v1.csv and C:\Users\ca\OneDrive\Documents\@ CA\@ PROJECTS\FFNNs\IDS 2026\data\raw\idi_claims_synth_v1.csv


## Reproducibility Checkpoint

In [9]:
# Reproducibility checkpoint
expo2, claims2 = generate_synth_idi_expo_claims(**PARAMS)

def df_hash(df: pd.DataFrame) -> str:
    b = df.to_csv(index=False).encode("utf-8")
    return sha256(b).hexdigest()

h1_expo, h2_expo = df_hash(expo_df), df_hash(expo2)
h1_clm, h2_clm = df_hash(claims_df), df_hash(claims2)

print("Exposure hash match:", h1_expo == h2_expo)
print("Claims   hash match:", h1_clm == h2_clm)
print("Exposure hash:", h1_expo[:16], "...")
print("Claims   hash:", h1_clm[:16], "...")

assert h1_expo == h2_expo and h1_clm == h2_clm, "Reproducibility check failed"

Exposure hash match: True
Claims   hash match: True
Exposure hash: 26d1f30cfdb351fc ...
Claims   hash: c4c10c85fcf9e082 ...
